Antes Preparacion del modelo

In [ ]:
# PSEUDOCODIGO:
# Preparacion, entrenamiento y validacion del modelo

# Libreria del modelo de deteccion de objetos (localiza la matricula en la imagen)
from yolov8 import YOLOv8
# Libreria de OCR (lee los caracteres de la matricula ya recortada)
from ocr_library import PaddleOCR
# Libreria numerica, util para manejar imagenes como matrices de numeros
import numpy as np

# Paso 1: Cargar dataset de imagenes anotadas
def preparar_dataset():
    # Lee todas las imagenes ficticias del banco de pruebas (sin personas identificables)
    imagenes = cargar_imagenes_desde_banco_datos('dataset_matriculas/')
    # Lee la posicion y el texto de la matricula de cada imagen (la "respuesta correcta")
    anotaciones = cargar_anotaciones('labels.json')
    # Dividir en train, val, test (80%, 10%, 10%)
    # train: para aprender | val: para ajustar | test: para la nota final, sin haberlo visto
    train_set, val_set, test_set = dividir_dataset(imagenes, anotaciones)
    # Devuelve los tres conjuntos para usarlos en los pasos siguientes
    return train_set, val_set, test_set

# Paso 2: Preprocesamiento de imagenes
def preprocesar_imagen(imagen):
    # El modelo exige un tamano fijo de entrada: 640 x 640 pixeles
    imagen_redimensionada = redimensionar(imagen, (640, 640))
    # Escala los valores de los pixeles a un rango pequeno (por ejemplo 0 a 1) para que el modelo aprenda mejor
    imagen_normalizada = normalizar(imagen_redimensionada)
    # Imagen lista para entrar al modelo
    return imagen_normalizada

# Paso 3: Entrenamiento del modelo
def entrenar_modelo(train_set):
    # Partimos de un modelo ya entrenado (yolov8n = version pequena): ahorra datos y tiempo
    modelo = YOLOv8('yolov8n.pt')  # Modelo preentrenado
    # Una epoca = una pasada completa por todo el conjunto de entrenamiento
    for epoca in range(50):
        # Se recorre el conjunto de entrenamiento por lotes (grupos de imagenes)
        for lote in train_set:
            # Misma preparacion que se aplicara despues en la aplicacion real
            imagen_preprocesada = preprocesar_imagen(lote['imagen'])
            # El modelo intenta localizar la matricula
            prediccion = modelo.predecir(imagen_preprocesada)
            # Mide cuanto se equivoca comparando con la etiqueta real
            perdida = calcular_perdida(prediccion, lote['etiqueta'])
            # Corrige el modelo para equivocarse menos la proxima vez
            actualizar_pesos(modelo, perdida)
    # Devuelve el modelo ya ajustado
    return modelo

# Paso 4: Validacion del modelo
def validar_modelo(modelo, val_set):
    # Aqui se guardara la seguridad (confianza) de cada prediccion
    confianzas = []
    # Se usan imagenes que el modelo no ha visto al entrenar
    for lote in val_set:
        imagen_preprocesada = preprocesar_imagen(lote['imagen'])
        prediccion = modelo.predecir(imagen_preprocesada)
        # Extrae de 0 a 1 lo seguro que esta el modelo de su respuesta
        confianza = obtener_confianza(prediccion)
        confianzas.append(confianza)
    # Resume todas las confianzas en un unico numero (por ejemplo, la media)
    precision_promedio = calcular_metrica(confianzas)
    # Sirve para decidir si el modelo es suficientemente bueno
    return precision_promedio

# Paso 5: Guardar modelo entrenado
def guardar_modelo(modelo):
    # Guarda el modelo en un fichero para no tener que entrenarlo cada vez (v1 = primera version)
    modelo.guardar('modelo_matriculator_v1.pt')
    print('Modelo guardado exitosamente')

# Ejecutar pipeline de preparacion
# 1. Prepara los datos
train_set, val_set, test_set = preparar_dataset()
# 2. Entrena con el conjunto de entrenamiento
modelo_entrenado = entrenar_modelo(train_set)
# 3. Mide la calidad con el conjunto de validacion
precision = validar_modelo(modelo_entrenado, val_set)
# 4. Guarda el resultado para usarlo en la aplicacion
guardar_modelo(modelo_entrenado)
# Muestra la precision como porcentaje con 2 decimales (por ejemplo, 92.50%)
print(f'Precision en validacion: {precision:.2%}')

Despues preparacion modelo

In [ ]:
# PSEUDOCODIGO: DESPUES de integrar el modelo en la aplicacion
# Flujo de inferencia cuando el usuario interactua con la aplicacion

# Mismo modelo de deteccion que se entreno antes
from yolov8 import YOLOv8
# Libreria de OCR para leer el texto de la matricula
from ocr_library import PaddleOCR

# Paso 1: Recepcion de imagen desde interfaz web
def recibir_imagen(archivo_usuario):
    # Convierte el fichero subido por la persona en una imagen que el programa pueda tratar
    imagen = cargar_imagen(archivo_usuario)
    return imagen

# Paso 2: Validacion de formato
def validar_formato(imagen):
    # Se intenta validar; si una condicion falla, salta al bloque except
    try:
        # Solo se aceptan JPG y PNG
        assert imagen.formato in ['JPG', 'PNG']
        # Limite de tamano para evitar ficheros enormes (10 MB)
        assert imagen.tamano < 10_MB
        # Si pasa las dos comprobaciones, el fichero es valido
        return True
    except AssertionError:
        # Alguna condicion ha fallado: el fichero no es valido
        return False  # Enviar mensaje error al usuario

# Paso 3: Preprocesamiento
def preprocesar_imagen(imagen):
    # Mismo tamano de entrada que en el entrenamiento (si no, el modelo rinde peor)
    imagen_redimensionada = redimensionar(imagen, (640, 640))
    # Misma escala de valores que en el entrenamiento
    imagen_normalizada = normalizar(imagen_redimensionada)
    return imagen_normalizada

# Paso 4: Cargar modelo entrenado
def cargar_modelo():
    # Carga el fichero guardado al entrenar (no se vuelve a entrenar)
    modelo = YOLOv8.cargar('modelo_matriculator_v1.pt')
    # Prepara el OCR; use_gpu=True lo acelera si hay tarjeta grafica disponible
    ocr = PaddleOCR(use_gpu=True)
    # Devuelve los dos componentes necesarios para leer la matricula
    return modelo, ocr

# Paso 5: Inferencia (deteccion + OCR)
def inferencia(imagen_preprocesada, modelo, ocr):
    # Deteccion de matricula (bounding box)
    # El modelo devuelve la lista de zonas donde cree que hay matriculas
    detecciones = modelo.predecir(imagen_preprocesada)
    # Nos quedamos con la primera (la mas probable)
    caja_matricula = detecciones[0]  # Coordenadas x1, y1, x2, y2
    # Seguridad del modelo al localizar la matricula (de 0 a 1)
    confianza_deteccion = detecciones.confianza

    # Extraer region de matricula
    # Recorta solo la zona de la matricula: filas de y1 a y2, columnas de x1 a x2
    matricula_crop = imagen_preprocesada[caja_matricula.y1:caja_matricula.y2, caja_matricula.x1:caja_matricula.x2]

    # OCR para leer caracteres
    # Convierte la imagen recortada en texto
    texto_matricula = ocr.leer_texto(matricula_crop)
    # Seguridad del OCR al leer los caracteres (de 0 a 1)
    confianza_ocr = ocr.confianza

    # Resultado en formato clave-valor (se enviara como JSON a la interfaz)
    return {
        # Texto leido, por ejemplo "1234 ABC"
        'matricula': texto_matricula,
        # Se toma la menor de las dos: la cadena es tan fiable como su eslabon mas debil
        'confianza': min(confianza_deteccion, confianza_ocr),
        # Posicion de la matricula, util para dibujarla en la imagen o revisarla
        'caja': caja_matricula
    }

# Paso 6: Evaluacion de confianza
def evaluar_confianza(resultado):
    # Umbral fijado al evaluar el modelo: por debajo, se pide revision humana
    umbral = 0.85  # 85%
    # True si el resultado es suficientemente fiable; False si no
    return resultado['confianza'] >= umbral

# Paso 7: Bifurcacion segun confianza
def procesar_resultado(resultado, confianza_valida):
    if confianza_valida:
        # Registro automatico
        # Se guarda sin intervencion humana, marcado como automatico
        guardar_registro(resultado, automatico=True)
        mensaje = f'Matricula detectada: {resultado["matricula"]}'
    else:
        # Derivar a revision humana
        # Se guarda como pendiente para que una persona lo compruebe y corrija
        guardar_pendiente_revision(resultado)
        mensaje = 'Confianza baja. Requiere revision manual.'
    # Mensaje que se mostrara en la pantalla de la persona usuaria
    return mensaje

# Paso 8: Salida final
def mostrar_resultado(mensaje, resultado):
    # Envia la respuesta a la interfaz web (en formato JSON)
    enviar_respuesta_web({
        'mensaje': mensaje,
        # .get evita un error si no hay matricula (por ejemplo, cuando el fichero no es valido)
        'matricula': resultado.get('matricula'),
        # Convierte 0.9231 en 92.31 para mostrarlo como porcentaje
        'confianza': round(resultado['confianza'] * 100, 2)
    })

# FLUJO PRINCIPAL
# 1. Entrada: la persona sube una imagen
imagen = recibir_imagen(archivo_usuario)

# 2. Control de errores: si el fichero no es valido, se avisa y se termina
if not validar_formato(imagen):
    mostrar_resultado('Error: formato no valido', {})
else:
    # 3. Prepara la imagen igual que en el entrenamiento
    imagen_prep = preprocesar_imagen(imagen)
    # 4. Carga el modelo y el OCR
    modelo, ocr = cargar_modelo()
    # 5. Detecta la matricula y la lee
    resultado = inferencia(imagen_prep, modelo, ocr)
    # 6. Comprueba si el resultado es suficientemente fiable
    confianza_ok = evaluar_confianza(resultado)
    # 7. Registra automaticamente o deriva a revision humana
    mensaje = procesar_resultado(resultado, confianza_ok)
    # 8. Salida: devuelve el mensaje y la matricula a la interfaz
    mostrar_resultado(mensaje, resultado)